In [ ]:
Cell 1: Load Clean Dataset
Pythonimport pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("data/processed/final_modeling_table_clean.csv")
print(f"Loaded clean dataset: {df.shape}")
Cell 2: Create Target Variable
Pythonnp.random.seed(42)

# Create realistic repayment label based on key risk factors
df['repayment_score'] = (
    (df['yield_kg_ha'] > df['yield_kg_ha'].quantile(0.4)) * 0.30 +
    (df['tx_frequency'] > 6) * 0.25 +
    (df['input_purchase_ratio'] > 0.12) * 0.20 +
    (df['soil_quality_index'] > 0.6) * 0.15 +
    (df['irrigated'] == 1) * 0.10
)

df['repayment_prob'] = df['repayment_score'].clip(0.15, 0.92)
df['repayment_label'] = (np.random.rand(len(df)) < df['repayment_prob']).astype(int)

print("Target Distribution:")
print(df['repayment_label'].value_counts(normalize=True))
Cell 3: Prepare Features
Pythonfeature_cols = [
    'farm_size', 'irrigated', 'inorganic_fertilizer', 'livestock',
    'yield_kg_ha', 'rainfall_mm', 'soil_quality_index', 'drought_risk',
    'climate_stress_index', 'tx_frequency', 'total_volume', 
    'avg_transaction', 'cashflow_volatility', 'input_purchase_ratio', 
    'fraud_rate', 'fertilizer_per_ha', 'input_efficiency'
]

# Use only available columns
available_features = [col for col in feature_cols if col in df.columns]
X = df[available_features].fillna(0)
y = df['repayment_label']

print(f"Modeling with {X.shape[1]} features")
Cell 4: Train Models
PythonX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# 1. Logistic Regression (Interpretable)
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
log_pred_prob = logreg.predict_proba(X_test)[:, 1]
print("Logistic Regression AUC-ROC:", round(roc_auc_score(y_test, log_pred_prob), 4))

# 2. XGBoost (Higher Performance)
xgb_model = xgb.XGBClassifier(n_estimators=300, learning_rate=0.08, max_depth=6, 
                              subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred_prob = xgb_model.predict_proba(X_test)[:, 1]
print("XGBoost AUC-ROC:", round(roc_auc_score(y_test, xgb_pred_prob), 4))



Run the notebook and share the AUC scores you get.
After that, we can:

Add SHAP explainability
Build the Streamlit dashboard
Generate the final deliverables